## What is Unsloth?

Unsloth is an open-source framework designed to accelerate the fine-tuning and inference of Large Language Models (LLMs). It specializes in optimizing Parameter-Efficient Fine-Tuning (PEFT) techniques—specifically Low-Rank Adaptation (LoRA) and its quantized variants (QLoRA). The library integrates directly with the Hugging Face ecosystem, including Transformers, TRL, and PEFT, allowing for rapid deployment of fine-tuned models for local execution or API integration.

## Why Unsloth is Faster

Unsloth achieves training speedups of 2x to 5x and reduces VRAM consumption by 40% to 80% with zero degradation in model accuracy. These performance gains are driven by exact mathematical optimizations rather than architectural approximations.

### Core Optimizations

| Optimization | Target Bottleneck | Mechanism | Result |
| --- | --- | --- | --- |
| **Kernel Fusion** | HBM read/write latency | Combines operations (RoPE, MLP) in SRAM via Triton | Reduces peak VRAM |
| **Manual Autograd** | Caching forward activations | Explicit mathematical gradient derivation | Lowers memory footprint |
| **Auto-Packing** | Padding token computation | Concatenates sequences into a 1D tensor | Increases tokens/second throughput |
| **Split LoRA** | Matrix multiplication overhead | Reorders operations using mathematical associativity | Accelerates MoE and LoRA updates |

---

### 1. Custom Triton Kernels and Operator Fusion

Standard PyTorch operations execute sequentially, materializing large intermediate tensors in High-Bandwidth Memory (HBM). Unsloth rewrites core modules—such as Rotary Positional Embeddings (RoPE), Multi-Layer Perceptrons (MLP), and the Cross-Entropy loss function—directly in OpenAI's Triton.

* **Logit Bottleneck Elimination:** During cross-entropy calculation, predicting the next token requires projecting the hidden state into a vocabulary space of size $V$. For large vocabularies, storing this $O(V)$ tensor consumes massive VRAM. Unsloth utilizes an online softmax algorithm, performing block-wise reductions across the hidden dimension $D$ and vocabulary $V$. This computes the scalar loss without materializing the full logit tensor.

### 2. Manual Autograd Derivation

PyTorch's automatic differentiation caches all intermediate forward activations to compute gradients during the backward pass.

* Unsloth bypasses this standard behavior by manually deriving the exact backpropagation steps for targeted layers. By explicitly defining the gradient functions in the framework, the engine avoids caching redundant activations during the forward pass. This significantly reduces the memory footprint required to store the computation graph.

### 3. Padding-Free Auto-Packing

Standard batched training requires padding sequences to match the length of the longest sequence in the batch, wasting compute cycles on mathematically irrelevant padding tokens.

* Unsloth addresses this by concatenating multiple independent sequences into a single, continuous, one-dimensional tensor. The Triton kernels are rewritten to handle variable-length indexing, ensuring that attention mechanisms respect individual sequence boundaries natively without calculating attention over padded regions. This exact mathematical restructuring yields speedups proportional to the dataset's sequence length variance.

### 4. Matrix Multiplication Associativity (Split LoRA)

For standard LoRA and Mixture of Experts (MoE) architectures, applying low-rank matrices ($A$ and $B$) to a pre-trained weight matrix ($W$) involves computing $Wx + B(Ax)$.

* Unsloth reorganizes these matrix operations by exploiting floating-point associativity. By restructuring the sequence of grouped Generalized Matrix Multiplications (GEMMs) alongside `torch._grouped_mm`, Unsloth minimizes computational complexity and memory usage during both the forward and backward passes.

In [ ]:
# Install Unsloth and necessary tools
!pip install unsloth
!pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 77.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
ERROR: Operation cancelled by user
^C


In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
max_seq_length = 2048 # You can increase this, Unsloth handles RoPE scaling automatically
dtype = None # None auto-detects (Float16 for T4)
load_in_4bit = True # Use 4-bit quantization to reduce memory usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.5.8: Fast Gemma2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Unsloth: Will load unsloth/gemma-2-2b-bnb-4bit as a legacy tokenizer.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: 8, 16, 32, 64 are standard
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Unsloth optimizes dropout = 0
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Enables long context with less VRAM
    random_state = 3407,
)

Unsloth 2026.5.8 patched 26 layers with 26 QKV layers, 26 O layers and 26 MLP layers.


In [ ]:
# Define the formatting template
prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a SQL query to answer the following question: {question}

### Input:
{context}

### Response:
{sql}"""

EOS_TOKEN = tokenizer.eos_token # Critical: prevents endless generation

def formatting_prompts_func(examples):
    questions = examples["sql_prompt"]
    contexts  = examples["sql_context"]
    sqls      = examples["sql"]
    texts = []

    for q, c, s in zip(questions, contexts, sqls):
        # Format the text and append the EOS token
        text = prompt_template.format(question=q, context=c, sql=s) + EOS_TOKEN
        texts.append(text)

    return { "text" : texts }

# Load the dataset from Hugging Face
dataset = load_dataset("philschmid/gretel-synthetic-text-to-sql", split = "train")

# Map the formatting function over the dataset
formatted_dataset = dataset.map(formatting_prompts_func, batched = True)

README.md:   0%|          | 0.00/737 [00:00<?, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…):   0%|          | 0.00/32.4M [00:00<?, ?B/s]

synthetic_text_to_sql_test.snappy.parque(…):   0%|          | 0.00/1.90M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Set True for faster training on short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 300, # Change to num_train_epochs = 1 for a full run which will take 2-3 days
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Start training
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Step,Training Loss
1,3.898509
2,3.713462
3,3.328432
4,3.236741
5,3.813987
6,3.401501
7,3.126389
8,3.411840
9,3.490119
10,3.462911


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-300/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-300.


In [ ]:
FastLanguageModel.for_inference(model) # Enables 2x faster inference

test_question = "What is the average trade value for each trader?"
test_context = "CREATE TABLE trade_history (id INT, trader_id INT, stock VARCHAR(255), price DECIMAL(5,2), quantity INT, trade_time TIMESTAMP);"

# Format our test prompt (leaving the SQL part blank)
test_prompt = prompt_template.format(
    question=test_question,
    context=test_context,
    sql=""
)

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")

# Generate the SQL output
outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=64) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a SQL query to answer the following question: What is the average trade value for each trader?

### Input:
CREATE TABLE trade_history (id INT, trader_id INT, stock VARCHAR(255), price DECIMAL(5,2), quantity INT, trade_time TIMESTAMP);

### Response:

































































